# Pre-processing News Data

## Bài toán
Dữ liệu gồm n văn bản phân vào 10 chủ đề khác nhau. Cần biễu diễn mỗi văn bản dưới dạng một vector số thể hiện cho nội dụng của văn bản đó.

## Mục lục
- Load dữ liệu từ thư mục
- Loại bỏ các stop words
- Sử dụng thư viện để mã hóa TF-IDF cho mỗi văn bản

In [ ]:
!pip install pyvi gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.9 MB/s eta 0:00:00


In [ ]:
# Based on the logs, gdown downloaded some data into 'data/news_vnexpress/news_vnexpress'
# We need to verify the path and load the files correctly.
import os
from sklearn.datasets import load_files

# Adjusting the input path based on the successful part of the download
INPUT_PATH = 'data/news_vnexpress/news_vnexpress'

if os.path.exists(INPUT_PATH):
    data_train = load_files(container_path=INPUT_PATH, encoding='utf-8')
    print(f"Data loaded successfully. Total documents: {len(data_train.filenames)}")
else:
    print(f"Path {INPUT_PATH} not found. Please ensure data is uploaded to this directory.")

Path data/news_vnexpress/news_vnexpress not found. Please ensure data is uploaded to this directory.


In [ ]:
!pip install pyvi gdown
import gdown
import os
import shutil
from sklearn.datasets import load_files

url = 'https://drive.google.com/drive/folders/1gNHur8cpJ1OeLHkxO9xtV0HvVfUZdyoB?usp=sharing'
os.makedirs('data/news_vnexpress', exist_ok=True)
try:
    gdown.download_folder(url, output='data/news_vnexpress', quiet=False)
except Exception as e:
    print(f'Download Note: {e}')

# Re-organize crawl_data if it contains individual category files
# load_files expects: path/category_name/file.txt
RAW_DATA = 'data/news_vnexpress/crawl_data'
PROCESSED_DATA = 'data/news_vnexpress/organized_data'

if os.path.exists(RAW_DATA):
    if not os.path.exists(PROCESSED_DATA):
        os.makedirs(PROCESSED_DATA, exist_ok=True)

    for filename in os.listdir(RAW_DATA):
        if filename.endswith('.txt'):
            category = filename.replace('.txt', '')
            cat_dir = os.path.join(PROCESSED_DATA, category)
            os.makedirs(cat_dir, exist_ok=True)
            shutil.copy(os.path.join(RAW_DATA, filename), os.path.join(cat_dir, filename))

    data_train = load_files(container_path=PROCESSED_DATA, encoding='utf-8')
    print(f'\nSuccessfully loaded: {len(data_train.filenames)} documents across {len(data_train.target_names)} categories.')
else:
    INPUT = 'data/news_vnexpress/news_vnexpress'
    if os.path.exists(INPUT):
        data_train = load_files(container_path=INPUT, encoding='utf-8')
        print(f'\nSuccessfully loaded from news_vnexpress: {len(data_train.filenames)} documents.')
    else:
        print('\nData path not found. Please ensure files are in the data directory.')

Retrieving folder contents


Retrieving folder 14aghwB-_3eQUKgo5EoDOtN3TXwlU1eSl crawl_data
Processing file 131g4euwvJtsc9KIQLoLiYki4Lh8j9O4A doi-song.txt
Processing file 1sFfn9sXE0QnLopBrjLChkKOZJjAbgEf1 du-lich.txt
Processing file 1MwbyhQUYdr-gdwyrhKJgY5q9uPHpKTJ4 giai-tri.txt
Processing file 1FtcST6SDMMcvJcurlZMfCcD2Zh_Q6h2a giao-duc.txt
Processing file 1giC_OWArHM8nxXiIqq2mlCOGD1mynTeH khoa-hoc.txt
Processing file 1gFUb1wFV1tLGxzEoopL7F42buK-v94tV kinh-doanh.txt
Processing file 1kdMbLAWnnm2KpV60CCqnh2kSy0erPBvM phap-luat.txt
Processing file 1YbUjL0ZXaKqnUJgG-AxjMe4vrxNeQQfX suc-khoe.txt
Processing file 1L7Ws1CrZkuoquVK1WugRcus0HhkpMq4A the-thao.txt
Processing file 1qL_xxVSLmAq7PEMafJv2XlzYEJ4ZGDco thoi-su.txt
Retrieving folder 1zDC0ZvzhLJht32XD2EWUSp0KLGEEGt7K news_vnexpress
Retrieving folder 1yOpAEPmkY5CFBb15jF7ie4kboht5nEVM doi-song
Processing file 1XzZahZ6CFh2IqMSp46wQwrNM600EEO3c 00000.txt
Processing file 1CbDH3kxAFvOZPxkiGzFtbh_8utl87Hci 00001.txt
Processing file 13vgishCHYiugcm4tohEwgA9sr7M7EUie 00002.tx

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from pyvi import ViTokenizer
import os

# 1. Load stopwords
stop_words_path = 'data/stopwords.txt'
if os.path.exists(stop_words_path):
    with open(stop_words_path, 'r', encoding='utf-8') as f:
        stop_words = [line.strip() for line in f.readlines()]
else:
    # Fallback list
    stop_words = ['và', 'của', 'là', 'có', 'trong']

print(f"Số lượng stopwords: {len(stop_words)}")

# 2. Tokenizer function
def Vietnamese_Tokenizer(text):
    return ViTokenizer.tokenize(text)

# 3. Vectorization logic
# Ensure data_train is defined before running this cell
try:
    module_count_vector = CountVectorizer(stop_words=stop_words, tokenizer=Vietnamese_Tokenizer)
    module_tfidf_transformer = TfidfTransformer()

    X_counts = module_count_vector.fit_transform(data_train.data)
    X = module_tfidf_transformer.fit_transform(X_counts)
    Y = data_train.target

    print(f"\nSố lượng từ trong từ điển: {len(module_count_vector.vocabulary_)}")
    print(f"Kích thước dữ liệu sau khi xử lý: {X.shape}")
    print(f"Kích thước nhãn tương ứng: {Y.shape}")
except NameError as e:
    print(f"Error: {e}. Please ensure you have loaded the data into 'data_train' first.")

Số lượng stopwords: 5
Error: name 'data_train' is not defined. Please ensure you have loaded the data into 'data_train' first.


## Phương pháp mã hóa: TF-IDF
Cho tập gồm $n$ văn bản: $D = \{d_1, d_2, ... d_n\}$. Tập từ điển tương ứng được xây dựng từ $n$ văn bản này có độ dài là $m$
- Xét văn bản $d$ có $|d|$ từ và $t$ là một từ trong $d$. Mã hóa tf-idf của $t$ trong văn bản $d$ được biểu diễn:
\begin{equation}
    \begin{split}
        \text{tf}_{t, d} &= \frac{f_t}{|d|} \\
        \text{idf}_{t, d} &= \log\frac{n}{n_t}, \ \ \ \ n_t = |\{d\in D: t\in d\}| \\
        \text{tf-idf}_{t d} &= \text{tf}_{t, d} \times \text{idf}_{t, d}
    \end{split}
\end{equation}

- Khi đó văn bản $d$ được mã hóa là một vector $m$ chiều. Các từ xuất hiện trong d sẽ được thay bằng giá trị tf-idf tương ứng. Các từ không xuất hiện trong $d$ thì thay là 0

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import gdown
from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

## Load dữ liệu từ thư mục

Cấu trúc thư mục như sau

- data/news_vnexpress/

    - Kinh tế:
        - bài báo 1.txt
        - bài báo 2.txt
    - Pháp luật
        - bài báo 3.txt
        - bài báo 4.txt

In [ ]:
INPUT = 'data/news_vnexpress'
os.makedirs("images",exist_ok=True)  # thư mục lưu các các hình ảnh trong quá trình huấn luyện và đánh gía

In [ ]:
# statistics
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('----------------------------------------------')
n = 0
for label in os.listdir(INPUT):
    print(f'{label}: {len(os.listdir(os.path.join(INPUT, label)))}')
    n += len(os.listdir(os.path.join(INPUT, label)))

print('-------------------------')
print(f"Tổng số văn bản: {n}")

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
-------------------------
Tổng số văn bản: 0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# load data
data_train = load_files(container_path=INPUT, encoding="utf-8")
print('mapping:')
for i in range(len(data_train.target_names)):
    print(f'{data_train.target_names[i]} - {i}')

print('--------------------------')
print(data_train.filenames[0:1])
# print(data_train.data[0:1])
print(data_train.target[0:1])
print(data_train.data[0:1])

print("\nTổng số  văn bản: {}" .format( len(data_train.filenames)))

mapping:
--------------------------
[]
[]
[]

Tổng số  văn bản: 0


## Chuyển dữ liệu dạng text về ma trận (n x m) bằng TF-IDF

* Bạn cần viết đoạn mã tương ứng trong cell bên dưới. Theo các bước được gợi ý

In [ ]:
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import os
import numpy as np
import pandas as pd

# 1. Load stopwords
stop_words_path = 'data/stopwords.txt'
if os.path.exists(stop_words_path):
    with open(stop_words_path, 'r', encoding='utf-8') as f:
        stop_words = [line.strip() for line in f.readlines()]
else:
    # Fallback list
    stop_words = ['và', 'của', 'là', 'có', 'trong', 'với', 'cho']

# 2. Tokenization function
def Vietnamese_Tokenizer(text):
    return ViTokenizer.tokenize(text)

# 3. Vectorization logic
try:
    # Initialize Vectorizers
    module_count_vector = CountVectorizer(stop_words=stop_words, tokenizer=Vietnamese_Tokenizer)
    module_tfidf_transformer = TfidfTransformer()

    # Transform data
    X_counts = module_count_vector.fit_transform(data_train.data)
    X = module_tfidf_transformer.fit_transform(X_counts)
    Y = data_train.target

    print(f"--- Kết quả Xử lý ---")
    print(f"Số lượng từ trong từ điển: {len(module_count_vector.vocabulary_)}")
    print(f"Kích thước ma trận TF-IDF: {X.shape}")

    # Detailed Insights: Top words per category
    print('\n--- Insights Chi Tiết: Top từ theo chủ đề ---')
    features = module_count_vector.get_feature_names_out()

    for category_id, category_name in enumerate(data_train.target_names):
        row_indices = np.where(Y == category_id)[0]
        if len(row_indices) > 0:
            # Calculate mean TF-IDF weight for terms in this category
            category_tfidf = X[row_indices].mean(axis=0).A1
            top_word_indices = category_tfidf.argsort()[-5:][::-1]

            print(f"\nChủ đề: {category_name.upper()}")
            for idx in top_word_indices:
                print(f"  - {features[idx]}: {category_tfidf[idx]:.4f}")

except NameError as e:
    print(f"Lỗi: {e}. Vui lòng chạy cell d0e8ed96 thành công trước.")

Lỗi: name 'data_train' is not defined. Vui lòng chạy cell d0e8ed96 thành công trước.


In [ ]:
print(X[100].toarray())
print(Y[100])

NameError: name 'X' is not defined

In [ ]:
sum(sum(X[100].toarray() != 0))

289

In [ ]:
print(X[100])

  (0, 12794)	0.14048828324700804
  (0, 12724)	0.051226678060487627
  (0, 12714)	0.034379239518190156
  (0, 12705)	0.024927343279465615
  (0, 12697)	0.03935911209707954
  (0, 12692)	0.013885134230282647
  (0, 12691)	0.02076954755505395
  (0, 12672)	0.03173992101554847
  (0, 12646)	0.04268947993761032
  (0, 12643)	0.030193779677554416
  (0, 12629)	0.024173036345759045
  (0, 12626)	0.01928809379275951
  (0, 12624)	0.3318224864003995
  (0, 12617)	0.08000423234784886
  (0, 12591)	0.07519534686809994
  (0, 12584)	0.03876774373554222
  (0, 12566)	0.033240367004725005
  (0, 12558)	0.03206234356763185
  (0, 12547)	0.04575286598942787
  (0, 12535)	0.05488370325838488
  (0, 12521)	0.09355442947181113
  (0, 12517)	0.03883219864696093
  (0, 12509)	0.017786174579851665
  (0, 12454)	0.07589970050190288
  (0, 12272)	0.02125953768208212
  :	:
  (0, 2170)	0.029508397725910254
  (0, 2159)	0.016084504788746505
  (0, 2140)	0.015661963686282587
  (0, 2135)	0.04322051581452054
  (0, 2111)	0.02577563465433511